# Audit v2 — Phase 2 data audit for *Leakage-Audited Solar Flare Forecasting Across Solar Cycles 24 and 25*

Avinash Kumar (ORCID 0009-0008-7613-1185). This notebook does **not** change the published v1.0.0 results; it checks the data and labels behind them.

**What it checks**
1. **GOES scaling.** Cycle 24 labels come from the SWAN-SF flare list (operational GOES classes, believed to carry the SWPC 0.7 scaling); Cycle 25 labels come from the NCEI GOES-16 science flare summary (true flux, no scaling). The notebook measures the ratio directly on overlapping flares.
2. **Label variants.** It rebuilds the windows once and labels them four ways: `v1` (as published); `v2a` (Cycle 25 threshold raised by the GOES-16/SWAN-SF flux ratio measured in Audit 1); `v2a_noaa` (same, with NOAA's documented 1/0.7); `v2b` (both cycles from the NOAA **GOES Flare Report**, one science-quality catalogue for GOES-8 to 19 in true flux, with SWPC active-region numbers).
3. **HEK region matching.** How often a flare has several HEK candidates with different NOAA regions, what the v1 `max` rule picks versus the nearest-in-time rule, and sensitivity to a ±5/10/20 minute window.
4. **HARP versus NOAA region.** How many HARPs carry several NOAA regions (and the reverse), and how many test HARPs share a NOAA region with a training HARP in each split.
5. **Recurrent regions (approximate).** HARP pairs that look like the same region returning after one rotation, across each train/test boundary.
6. **Test date ranges.** The real first and last date of every train and test set.

*Run 2 changes:* the GOES-15 flare-summary path used in run 1 does not exist; Cycle 24 true-flux labels now come from the GOES Flare Report. Each input is downloaded in its own step, so one missing file no longer blocks the HEK download.

**How to run (Google Colab, CPU is enough):** `Runtime → Run all`. If you have the Drive backup from v1 (`MyDrive/solar_bench`), it is reused and the run takes about 30–45 minutes; from scratch it takes about 1.5–2 hours. Every result is written to `audit_v2_out/`, zipped, copied to `MyDrive/solar_bench_v2/` and offered as a download at the end.

Each audit cell catches its own errors, so one failure does not stop the others; the error is printed and recorded in `audit_v2_out/errors.txt`.

In [1]:
!pip -q install drms netCDF4 xarray pyarrow


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, re, io, json, time, glob, shutil, tarfile, hashlib, platform, traceback, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

WORK = os.environ.get("AUDIT_WORK", "/content")
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
for d in ["swan", "pinn_data", "hek_cache", "audit_v2_out"]:
    os.makedirs(d, exist_ok=True)
OUT = "audit_v2_out"
BACKUP_V1 = "/content/drive/MyDrive/solar_bench"
BACKUP_V2 = "/content/drive/MyDrive/solar_bench_v2"

P = ['ABSNJZH','MEANALP','MEANGAM','MEANGBH','MEANGBT','MEANGBZ','MEANJZD','MEANJZH',
     'MEANPOT','MEANSHR','R_VALUE','SAVNCPP','SHRGT45','TOTPOT','TOTUSJH','TOTUSJZ','USFLUX']
W = 12
SCALE = 0.7                    # SWPC scaling of GOES 8-15 operational XRS-B (NOAA GOES-R XRS L2 Users Guide)
M_TRUE = 1e-5                  # M1.0 in true flux
M_SCALED_EQ = 1e-5 / SCALE     # operational M1.0 expressed in true flux (~M1.43)
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36"}

ERRORS = []
def run_audit(name, fn):
    """Run one audit; print and record any error instead of stopping the notebook."""
    t0 = time.time(); print(f"\n========== {name} ==========", flush=True)
    try:
        fn(); print(f"[{name}] done in {round(time.time()-t0)} s", flush=True)
    except Exception:
        tb = traceback.format_exc(); ERRORS.append(f"## {name}\n{tb}")
        print(f"[{name}] FAILED — recorded in {OUT}/errors.txt\n{tb}", flush=True)
    with open(f"{OUT}/errors.txt", "w") as f:
        f.write("\n".join(ERRORS) if ERRORS else "no errors\n")

def save(df, name, note=""):
    path = f"{OUT}/{name}.csv"; df.to_csv(path, index=False)
    print(f"--> saved {path}" + (f"  ({note})" if note else "")); print(df.to_string(index=False)); return df

def get(url, path):
    if os.path.exists(path): print("have", path); return
    import requests
    with requests.get(url, headers=UA, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(path + ".part", "wb") as f:
            for ch in r.iter_content(1 << 20): f.write(ch)
    os.replace(path + ".part", path)
    print("saved", path, round(os.path.getsize(path)/1e6, 1), "MB", flush=True)

def cls2flux(s):
    """GOES class string -> peak XRS-B flux in W/m^2 (same convention as the class)."""
    m = re.match(r"^\s*([ABCMX])\s*([0-9]*\.?[0-9]+)?", str(s).upper())
    if not m: return np.nan
    return {"A": 1e-8, "B": 1e-7, "C": 1e-6, "M": 1e-5, "X": 1e-4}[m.group(1)] * float(m.group(2) or 1.0)

def to_time(s):
    return pd.to_datetime(s.astype(str).str.replace("_TAI", "").str.replace(r"^(\d{4})\.(\d\d)\.(\d\d)_", r"\1-\2-\3 ", regex=True), errors="coerce")

import sklearn
vers = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__}
for mod in ["drms", "xarray", "netCDF4"]:
    try: vers[mod] = __import__(mod).__version__
    except Exception: vers[mod] = "not installed"
print(vers)

{'python': '3.12.15', 'pandas': '3.1.0', 'numpy': '2.5.4', 'scikit-learn': '1.9.1', 'drms': '0.9.1', 'xarray': '2026.9.0', 'netCDF4': '1.7.5.1.post1'}


## 0. Restore Drive backups (v1 cache and any earlier v2 run)

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for src in [BACKUP_V1, BACKUP_V2]:
        if not os.path.isdir(src): print("no backup at", src); continue
        n = 0
        for root, _, fs in os.walk(src):
            for f in fs:
                rel = os.path.relpath(os.path.join(root, f), src)
                if rel.startswith(OUT) or os.path.exists(rel): continue
                os.makedirs(os.path.dirname(rel) or ".", exist_ok=True); shutil.copy(os.path.join(root, f), rel); n += 1
        print("restored", n, "files from", src)
except Exception as e:
    print("restore skipped:", type(e).__name__)

restore skipped: ModuleNotFoundError


## 1. Download inputs (skips anything already on disk)

In [4]:
import requests

def dl_swan():
    get("https://dataverse.harvard.edu/api/access/datafile/3642452", "swan/addenda.tar.gz")
    if not os.path.exists("swan/addenda/integrated_flare_data"):
        with tarfile.open("swan/addenda.tar.gz") as t: t.extractall("swan/addenda", filter="data")

def dl_sharp():
    import drms
    KEYS = ("HARPNUM, T_REC, NOAA_ARS, NOAA_NUM, QUALITY, LAT_FWT, LON_FWT, CRLN_OBS, "
            "USFLUX, MEANGAM, MEANGBT, MEANGBZ, MEANGBH, MEANJZD, TOTUSJZ, MEANALP, MEANJZH, "
            "TOTUSJH, ABSNJZH, SAVNCPP, MEANPOT, TOTPOT, MEANSHR, SHRGT45, R_VALUE, AREA_ACR, SIZE_ACR, NACR")
    def jsoc_months(months, out):
        if os.path.exists(out): print("have", out); return
        c = drms.Client(); parts = []
        for m in months:
            start = m.strftime("%Y.%m.%d_00:00:00_TAI")
            for a in range(3):
                try: ks = c.query(f"hmi.sharp_cea_720s[][{start}/{m.days_in_month}d@1h]", key=KEYS); break
                except Exception as e: print("retry", m.date(), type(e).__name__); time.sleep(5)
            else: raise RuntimeError(f"JSOC failed {m.date()}")
            parts.append(ks)
            if m.month == 12: print(m.year, "done", flush=True)
        pd.concat(parts, ignore_index=True).to_csv(out, index=False)
    jsoc_months(pd.date_range("2019-01-01", "2024-12-01", freq="MS"), "pinn_data/sharp_2019_2024_1h.csv.gz")
    jsoc_months(pd.date_range("2010-05-01", "2018-12-01", freq="MS"), "pinn_data/sharp_2010_2018_1h.csv.gz")

GOES = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/"
def dl_g16():
    get(GOES + "goes16/l2/data/xrsf-l2-flsum_science/sci_xrsf-l2-flsum_g16_s20170209_e20250406_v2-2-1.nc", "flsum_g16.nc")

def dl_flrpt():
    """NOAA GOES Flare Report: GOES-8..19 science-quality flares (true flux) with SWPC active regions."""
    # yearly netCDF files have stable names (the mission-length file is renamed every day)
    d = GOES + "multi/l2/data/xrsf-l2-flrpt_science/nc/"
    try: listing = requests.get(d, headers=UA, timeout=120).text
    except Exception as e: listing = ""; print("listing failed:", type(e).__name__)
    os.makedirs("flrpt", exist_ok=True)
    for yr in range(2010, 2026):
        found = sorted(set(re.findall(rf"sci_xrsf-l2-flrpt_geo_y{yr}_v[0-9\-]+\.nc", listing)))
        name = found[-1] if found else f"sci_xrsf-l2-flrpt_geo_y{yr}_v1-0-1.nc"
        if not glob.glob(f"flrpt/sci_xrsf-l2-flrpt_geo_y{yr}_*.nc"): get(d + name, f"flrpt/{name}")
    print("Flare Report files:", sorted(os.listdir("flrpt")))

def dl_hek():
    """HEK flare records, all classes, cached per month, 2010-05 .. 2025-04."""
    url = "https://www.lmsal.com/hek/her"
    ret = "event_starttime,event_peaktime,fl_goescls,ar_noaanum,hpc_x,hpc_y,frm_name"
    for frm in ["SWPC", "SSW Latest Events"]:
        slug = frm.replace(" ", "_"); failed = []
        for m in pd.date_range("2010-05-01", "2025-04-30", freq="MS"):
            path = f"hek_cache/{slug}_{m:%Y-%m}.csv"
            if os.path.exists(path): continue
            p = {"cosec": 2, "cmd": "search", "type": "column", "event_type": "fl",
                 "event_starttime": m.strftime("%Y-%m-%dT00:00:00"), "event_endtime": (m + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%dT23:59:59"),
                 "event_coordsys": "helioprojective", "x1": -1200, "x2": 1200, "y1": -1200, "y2": 1200, "result_limit": 5000,
                 "param0": "FRM_Name", "op0": "=", "value0": frm, "return": ret}
            for k in range(4):
                try: df = pd.DataFrame(requests.get(url, params=p, headers=UA, timeout=180).json()["result"]); break
                except Exception: time.sleep(5)
            else: failed.append(f"{m:%Y-%m}"); continue
            if len(df) >= 5000: print("WARNING: HEK result limit reached", frm, f"{m:%Y-%m}")
            df.to_csv(path, index=False)
        print("HEK", frm, "cached;", "failed months: " + ", ".join(failed) if failed else "all months ok", flush=True)
        checkpoint()
        if failed: raise RuntimeError(f"HEK {frm}: {len(failed)} months failed (re-run this cell to retry): {failed}")

def checkpoint():
    """Copy slow-to-rebuild inputs to Drive now, so a Colab disconnect does not lose them."""
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
        os.makedirs(f"{BACKUP_V2}/hek_cache", exist_ok=True)
        os.makedirs(f"{BACKUP_V2}/flrpt", exist_ok=True)
        for f in glob.glob("hek_cache/*.csv") + glob.glob("flrpt/*.nc") + [p for p in ["flsum_g16.nc"] if os.path.exists(p)]:
            if not os.path.exists(f"{BACKUP_V2}/{f}"): shutil.copy(f, f"{BACKUP_V2}/{f}")
        print("checkpoint saved to", BACKUP_V2)
    except Exception as e: print("checkpoint skipped:", type(e).__name__)

for nm, fn in [("download SWAN-SF", dl_swan), ("download SHARP", dl_sharp), ("download GOES-16 flare summary", dl_g16),
               ("download GOES Flare Report", dl_flrpt), ("download HEK", dl_hek)]:
    run_audit(nm, fn)
checkpoint()


========== download SWAN-SF ==========


saved swan/addenda.tar.gz 34.0 MB


[download SWAN-SF] done in 2 s



========== download SHARP ==========


2019 done


...

HEK SWPC cached; all months ok


checkpoint skipped: ModuleNotFoundError


HEK SSW Latest Events cached; all months ok


checkpoint skipped: ModuleNotFoundError
[download HEK] done in 560 s


checkpoint skipped: ModuleNotFoundError


## 2. Load flare catalogues

In [5]:
FL, FLUX_SRC, FLRPT_VARS = {}, {}, []
def _txt(a):
    """1-D array of strings from a netCDF string/char variable."""
    a = np.asarray(a)
    if a.ndim == 2: return np.array([b"".join(x).decode().strip() if isinstance(x[0], bytes) else "".join(x).strip() for x in a])
    return np.array([x.decode().strip() if isinstance(x, bytes) else str(x).strip() for x in a])
def read_flrpt(path):
    """Only the 1-D variables we need (location variables are 2-D and would multiply rows)."""
    import xarray as xr
    ds = xr.open_dataset(path)
    if not FLRPT_VARS: FLRPT_VARS.extend(ds.variables); print(os.path.basename(path), "variables:", list(ds.variables))
    ar = ds["active_region"].values if "active_region" in ds.variables else np.zeros(ds["time"].size)
    if ar.dtype.kind in "SUO": ar = pd.to_numeric(pd.Series(_txt(ar)), errors="coerce").values
    return pd.DataFrame({"time": pd.to_datetime(ds["time"].values), "flare_class": _txt(ds["flare_class"].values),
                         "xrsb_irrad": ds["xrsb_irrad"].values.astype(float), "active_region": ar})
def load_catalogues():
    import xarray as xr
    g = pd.read_csv("swan/addenda/integrated_flare_data/goes_flares_integrated.csv", parse_dates=["peak_time"])
    g = g.rename(columns={"peak_time": "time", "goes_class": "cls"})
    g["cls"] = g.cls.astype(str).str.strip(); g["flux"] = g.cls.map(cls2flux)
    g["ar"] = pd.to_numeric(g.noaa_active_region, errors="coerce").fillna(0).astype(int)
    FL["swan"] = g[["time", "cls", "flux", "ar"]].dropna(subset=["time", "flux"]).sort_values("time").reset_index(drop=True)

    def load_flsum(path):
        d = xr.open_dataset(path).to_dataframe().reset_index()
        for c in d.columns:
            if d[c].dtype == object: d[c] = d[c].map(lambda v: v.decode() if isinstance(v, bytes) else v)
        print(path, "columns:", list(d.columns))
        d = d[d.status.astype(str).str.strip() == "EVENT_PEAK"].copy()
        d["time"] = pd.to_datetime(d["time"]); d["cls"] = d.flare_class.astype(str).str.strip()
        d["flux"] = d.cls.map(cls2flux)            # class-based (defines "M1.0+" exactly as the class string does)
        # continuous peak flux when the file has it; class strings are rounded to 0.1, which blurs the ~M1.43 cut
        pk = next((c for c in ["xrsb_flux", "peak_flux"] if c in d.columns), None)
        d["flux_pk"] = pd.to_numeric(d[pk], errors="coerce") if pk else d.flux
        FLUX_SRC[path] = pk or "class string"
        return d[["time", "cls", "flux", "flux_pk"]].dropna(subset=["time", "flux"]).sort_values("time").reset_index(drop=True)
    FL["g16"] = load_flsum("flsum_g16.nc")
    fr = sorted(glob.glob("flrpt/sci_xrsf-l2-flrpt_geo_y*.nc"))
    if fr:
        r = pd.concat([read_flrpt(f) for f in fr], ignore_index=True)
        r["cls"] = r.flare_class.astype(str).str.strip()
        r["flux"] = r.cls.map(cls2flux); r["flux_pk"] = pd.to_numeric(r.xrsb_irrad, errors="coerce")
        r["ar"] = pd.to_numeric(r.active_region, errors="coerce").fillna(0).astype(int)
        r.loc[(r.ar > 0) & (r.ar < 10000), "ar"] += 10000
        FL["flrpt"] = r[["time", "cls", "flux", "flux_pk", "ar"]].dropna(subset=["time", "flux"]).sort_values("time").reset_index(drop=True)

    parts = [pd.read_csv(f) for f in sorted(glob.glob("hek_cache/*.csv")) if os.path.getsize(f) > 2]
    cols = ["event_peaktime", "ar_noaanum", "fl_goescls", "frm_name"]
    hk = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)
    if not parts: print("WARNING: no HEK records on disk; v1/v2a Cycle 25 labels and Audit 3 need them")
    hk["peak"] = pd.to_datetime(hk.event_peaktime, errors="coerce")
    hk["ar"] = pd.to_numeric(hk.ar_noaanum, errors="coerce").fillna(0).astype(int)
    hk.loc[(hk.ar > 0) & (hk.ar < 10000), "ar"] += 10000
    hk["cls"] = hk.fl_goescls.astype(str).str.strip()
    FL["hek"] = hk.dropna(subset=["peak"])[["peak", "ar", "cls", "frm_name"]].sort_values("peak").reset_index(drop=True)
    rows = [dict(catalogue=k, n_events=len(v), first=str(v.iloc[:, 0].min()), last=str(v.iloc[:, 0].max()),
                 n_Mplus=int((v.cls.str[:1].isin(["M", "X"])).sum())) for k, v in FL.items()]
    save(pd.DataFrame(rows), "catalogues_loaded")

run_audit("load catalogues", load_catalogues)


========== load catalogues ==========


[load catalogues] FAILED — recorded in audit_v2_out/errors.txt
Traceback (most recent call last):
  File "/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/xarray/backends/file_manager.py", line 265, in _acquire_with_cache_info
    file = self._cache[self._key]
           ~~~~~~~~~~~^^^^^^^^^^^
  File "/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/xarray/backends/lru_cache.py", line 56, in __getitem__
    value = self._cache[key]
            ~~~~~~~~~~~^^^^^
KeyError: [<class 'netCDF4._netCDF4.Dataset'>, ('/home/runner/work/solar-flare-leakage-audit/solar-flare-leakage-audit/work/flsum_g16.nc',), 'r', (('clobber', True), ('diskless', False), ('format', 'NETCDF4'), ('persist', False)), 'f4aec3af-419f-4812-9c01-908f2c796abd']

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_2440/2526661432.py", line 26, in run_audit
    fn(); print(f"[{name}] done in {round(time.time()

## Audit 1 — GOES scaling: measure the flux ratio on flares present in two catalogues

In [6]:
def nearest_pairs(a, b, tol_min=5, on_a="time", on_b="time"):
    """For each row of a, the nearest row of b within tol (one-to-one not enforced)."""
    A = a.sort_values(on_a).reset_index(drop=True).rename(columns=lambda c: c + "_a")
    B = b.sort_values(on_b).reset_index(drop=True).rename(columns=lambda c: c + "_b")
    m = pd.merge_asof(A, B, left_on=on_a + "_a", right_on=on_b + "_b", direction="nearest", tolerance=pd.Timedelta(f"{tol_min}min"))
    return m.dropna(subset=[on_b + "_b"])

RATIO = {"g16_over_swan": 1 / SCALE}   # replaced by the measured value when Audit 1 has >= 30 matched flares
def audit_goes_scaling():
    rows = []
    def compare(name, a, b, t0, t1, note):
        a = a[(a.time >= t0) & (a.time <= t1) & (a.flux >= 5e-6)]; b = b[(b.time >= t0) & (b.time <= t1)]
        m = nearest_pairs(a, b, 5)
        r = (m.flux_pk_b / m.flux_a).replace([np.inf, -np.inf], np.nan).dropna()
        rows.append(dict(comparison=name, period=f"{t0}..{t1}", n_a_ge_C5=len(a), n_matched=len(r),
                         median_ratio_b_over_a=round(r.median(), 3) if len(r) else np.nan,
                         p10=round(r.quantile(.1), 3) if len(r) else np.nan, p90=round(r.quantile(.9), 3) if len(r) else np.nan, expect=note))
    s = FL["swan"]
    compare("G16 science / SWAN-SF", s, FL["g16"], "2017-02-09", "2018-12-31", "~1.43 if SWAN-SF is 0.7-scaled")
    if "flrpt" in FL:
        compare("Flare Report / SWAN-SF", s, FL["flrpt"], "2010-05-01", "2018-12-31", "~1.43 if SWAN-SF is 0.7-scaled")
        compare("Flare Report / G16 flsum", FL["g16"].assign(flux=FL["g16"].flux_pk), FL["flrpt"], "2019-01-01", "2024-12-31", "~1.0 if the report uses GOES-16 for Cycle 25")
    R = save(pd.DataFrame(rows), "audit1_goes_scaling", "median ratio of peak flux for the same flare")
    g = R[R.comparison == "G16 science / SWAN-SF"]
    if len(g) and g.n_matched.iloc[0] >= 30: RATIO["g16_over_swan"] = float(g.median_ratio_b_over_a.iloc[0])
    print("ratio used for v2a:", RATIO)
    big = []
    for k in ["swan", "g16", "flrpt"]:
        if k in FL:
            d = FL[k]; d = d[(d.time >= "2017-09-06 11:30") & (d.time <= "2017-09-06 12:30")]
            big.append(dict(catalogue=k, events=", ".join(f"{t:%H:%M} {c}" for t, c in zip(d.time, d.cls))))
    save(pd.DataFrame(big), "audit1_example_2017_09_06", "the X9.3 flare as listed in each catalogue")

run_audit("Audit 1 GOES scaling", audit_goes_scaling)


========== Audit 1 GOES scaling ==========


[Audit 1 GOES scaling] FAILED — recorded in audit_v2_out/errors.txt
Traceback (most recent call last):
  File "/tmp/ipykernel_2440/2526661432.py", line 26, in run_audit
    fn(); print(f"[{name}] done in {round(time.time()-t0)} s", flush=True)
    ^^^^
  File "/tmp/ipykernel_2440/2079520713.py", line 19, in audit_goes_scaling
    compare("G16 science / SWAN-SF", s, FL["g16"], "2017-02-09", "2018-12-31", "~1.43 if SWAN-SF is 0.7-scaled")
                                        ~~^^^^^^^
KeyError: 'g16'



## Audit 3 — HEK region matching (run before building labels; the labels use the chosen rule)

In [7]:
def hek_assign(fl, hk, win_min=10, mx_only=False):
    """For each flare (time) return HEK candidates within +-win, the v1 rule (max AR among candidates)
    and the v2 rule (AR of the nearest-in-time candidate that has an AR)."""
    h = hk[hk.cls.str[:1].isin(["M", "X"])] if mx_only else hk[hk.cls.str[:1].isin(["C", "M", "X"])]
    hp = h.peak.values.astype("datetime64[ns]"); har = h.ar.values; w = np.timedelta64(win_min, "m")
    out = []
    for t in fl.time.values.astype("datetime64[ns]"):
        i0, i1 = np.searchsorted(hp, t - w, "left"), np.searchsorted(hp, t + w, "right")
        ars, dts = har[i0:i1], np.abs(hp[i0:i1] - t)
        nz = ars > 0
        v1 = int(ars.max()) if len(ars) else 0
        v2 = int(ars[nz][np.argmin(dts[nz])]) if nz.any() else 0
        out.append((i1 - i0, len(set(ars[nz])), v1, v2))
    return pd.DataFrame(out, columns=["n_cand", "n_distinct_ar", "ar_v1max", "ar_v2nearest"], index=fl.index)

def audit_hek():
    rows = []
    sets = [("C25 G16 M+ (true flux)", FL["g16"][(FL["g16"].time >= "2019-01-01") & (FL["g16"].flux >= M_TRUE)])]
    if "flrpt" in FL: sets.append(("C24 Flare Report M+ (true flux)", FL["flrpt"][(FL["flrpt"].time >= "2010-05-01") & (FL["flrpt"].time <= "2018-12-31") & (FL["flrpt"].flux >= M_TRUE)]))
    for name, fl in sets:
        for win in [5, 10, 20]:
            for mx in ([True, False] if win == 10 else [False]):
                a = hek_assign(fl, FL["hek"], win, mx_only=mx)
                rows.append(dict(flares=name, window_min=win, hek_classes="M/X only (v1)" if mx else "C/M/X",
                                 n_flares=len(a), with_ar_v1=int((a.ar_v1max > 0).sum()), with_ar_v2=int((a.ar_v2nearest > 0).sum()),
                                 ambiguous_multi_ar=int((a.n_distinct_ar > 1).sum()),
                                 v1_v2_disagree=int(((a.ar_v1max != a.ar_v2nearest) & (a.ar_v1max > 0)).sum())))
    save(pd.DataFrame(rows), "audit3_hek_matching", "v1 = max AR among M/X HEK records within 10 min")
    if "flrpt" in FL:
        rows = []
        for cyc, t0, t1 in [(24, "2010-05-01", "2018-12-31"), (25, "2019-01-01", "2024-12-31")]:
            f = FL["flrpt"][(FL["flrpt"].time >= t0) & (FL["flrpt"].time <= t1) & (FL["flrpt"].flux >= M_TRUE)]
            h = hek_assign(f, FL["hek"], 10).ar_v2nearest.values
            row = dict(cycle=cyc, report_Mplus=len(f), report_with_ar=int((f.ar > 0).sum()), hek_ar_found=int((h > 0).sum()),
                       report_and_hek_same_ar=int(((f.ar.values == h) & (h > 0)).sum()))
            if cyc == 24:
                m = nearest_pairs(f[["time", "ar"]], FL["swan"][FL["swan"].flux >= 5e-6][["time", "ar"]], 5)
                row.update(matched_to_swan=len(m), report_and_swan_same_ar=int(((m.ar_a == m.ar_b) & (m.ar_a > 0)).sum()))
            rows.append(row)
        save(pd.DataFrame(rows), "audit3_ar_agreement", "NOAA region from the Flare Report vs HEK (and vs SWAN-SF for Cycle 24)")

run_audit("Audit 3 HEK matching", audit_hek)


========== Audit 3 HEK matching ==========


[Audit 3 HEK matching] FAILED — recorded in audit_v2_out/errors.txt
Traceback (most recent call last):
  File "/tmp/ipykernel_2440/2526661432.py", line 26, in run_audit
    fn(); print(f"[{name}] done in {round(time.time()-t0)} s", flush=True)
    ^^^^
  File "/tmp/ipykernel_2440/1288095453.py", line 18, in audit_hek
    sets = [("C25 G16 M+ (true flux)", FL["g16"][(FL["g16"].time >= "2019-01-01") & (FL["g16"].flux >= M_TRUE)])]
                                       ~~^^^^^^^
KeyError: 'g16'



## Audit 2 — Rebuild windows once and label them three ways

In [8]:
SAMPLES, HARPINFO = {}, {}
VARIANTS = ["v1", "v2a", "v2a_noaa", "v2b"]
def flare_table(variant, cycle):
    """Flares (time, ar) defining the M+ target for one variant and cycle; None if the input is missing."""
    if variant == "v2b":
        if "flrpt" not in FL: return None
        t0, t1 = ("2010-05-01", "2019-01-01") if cycle == 24 else ("2019-01-01", "2025-05-01")
        f = FL["flrpt"]; f = f[(f.time >= t0) & (f.time < t1) & (f.flux >= M_TRUE)]
        return f[f.ar > 0][["time", "ar"]]
    if cycle == 24:
        g = FL["swan"]; return g[(g.flux >= M_TRUE) & (g.ar > 0)][["time", "ar"]]
    f = FL["g16"][FL["g16"].time >= "2019-01-01"].copy()
    if variant == "v1":
        f = f[f.flux >= M_TRUE]; f["ar"] = hek_assign(f, FL["hek"], 10, mx_only=True).ar_v1max
    else:   # operational M1.0 expressed in GOES-16 true flux
        thr = 1e-5 * (RATIO["g16_over_swan"] if variant == "v2a" else 1 / SCALE)
        f = f[f.flux_pk >= thr] if FLUX_SRC.get("flsum_g16.nc", "class string") != "class string" else f[f.flux >= round(thr, 6) * (1 - 1e-9)]
        f["ar"] = hek_assign(f, FL["hek"], 10).ar_v2nearest
    return f[f.ar > 0][["time", "ar"]]

def build_cycle(cycle):
    path = "pinn_data/sharp_2010_2018_1h.csv.gz" if cycle == 24 else "pinn_data/sharp_2019_2024_1h.csv.gz"
    c = pd.read_csv(path); c["time"] = to_time(c.T_REC)
    c = c[c.QUALITY == 0].sort_values(["HARPNUM", "time"]).reset_index(drop=True)
    c["arset"] = c.NOAA_ARS.astype(str).apply(lambda s: [int(x) for x in re.findall(r"\b\d{5}\b", s)])
    hp_ars = c.groupby("HARPNUM").arset.apply(lambda L: set(a for l in L for a in l))
    keep = hp_ars[hp_ars.apply(len) > 0].index; c = c[c.HARPNUM.isin(keep)]
    span = c.groupby("HARPNUM").time.agg(["min", "max"])
    c["carr"] = (c.CRLN_OBS + c.LON_FWT) % 360
    info = c.groupby("HARPNUM").agg(t0=("time", "min"), t1=("time", "max"), lat=("LAT_FWT", "median"))
    info["carr0"] = c.groupby("HARPNUM").carr.first(); info["carr1"] = c.groupby("HARPNUM").carr.last()
    info["ars"] = hp_ars.loc[info.index]; HARPINFO[cycle] = info
    ar2h = {}
    for h in keep:
        for a in hp_ars[h]: ar2h.setdefault(a, []).append(h)
    events = {}
    for v in VARIANTS:
        try: ft = flare_table(v, cycle)
        except Exception as e: print(f"C{cycle} {v}: FAILED {type(e).__name__}: {e}"); ft = None
        if ft is None: print(f"C{cycle} {v}: skipped"); continue
        ev = {h: [] for h in keep}
        for t, a in zip(ft.time, ft.ar):
            for h in ar2h.get(int(a), []):
                if span.loc[h, "min"] <= t <= span.loc[h, "max"] + pd.Timedelta("24h"): ev[h].append(np.datetime64(t, "ns"))
        events[v] = {h: np.sort(np.array(e, dtype="datetime64[ns]")) for h, e in ev.items()}
    rows = []
    for h, d in c.groupby("HARPNUM"):
        d = d.reset_index(drop=True)
        if len(d) < W: continue
        X = d[P].astype("float64"); r = X.rolling(W, min_periods=9)
        f = pd.concat([X.add_suffix("_last"), r.mean().add_suffix("_mean"), r.std().add_suffix("_std"), X.diff(W-1).add_suffix("_delta")], axis=1)
        t = d.time.values.astype("datetime64[ns]")
        for v, E in events.items():
            e = E[h]; f["y_" + v] = (np.searchsorted(e, t + np.timedelta64(24, "h"), side="right") > np.searchsorted(e, t, side="right")).astype("int8")
        f["time"] = d.time; f["harp"] = h; f["ar"] = f"c{cycle}_{h}"
        rows.append(f.dropna(subset=[x + "_last" for x in P] + [x + "_mean" for x in P]).iloc[W-1:])
    S = pd.concat(rows, ignore_index=True)
    S = S[(S.time >= ("2010-06-01" if cycle == 24 else "2019-02-01")) & (S.time <= ("2018-12-30" if cycle == 24 else "2024-12-30"))]
    FEc = [x for x in S.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
    S[FEc] = S[FEc].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    S = S.reset_index(drop=True); S.to_pickle(f"c{cycle}_samples_v2.pkl"); SAMPLES[cycle] = S
    return S

def audit_labels():
    expect = {24: (215463, 7995), 25: (162801, 13289)}
    rows = []
    for cyc in [24, 25]:
        S = build_cycle(cyc)
        for v in VARIANTS:
            if "y_" + v not in S: continue
            ft = flare_table(v, cyc)
            rows.append(dict(cycle=cyc, variant=v, Mplus_flares_with_ar=len(ft), windows=len(S), positive_windows=int(S["y_" + v].sum()),
                             positive_rate=round(S["y_" + v].mean(), 4),
                             reproduces_v1=(len(S), int(S["y_" + v].sum())) == expect[cyc] if v == "v1" else ""))
        if "y_v2b" in S:
            ch = S[S.y_v1 != S.y_v2b]
            print(f"C{cyc}: windows whose label differs v1 vs v2b: {len(ch)} (0->1: {int((ch.y_v2b == 1).sum())}, 1->0: {int((ch.y_v2b == 0).sum())})")
    save(pd.DataFrame(rows), "audit2_label_variants",
         "v1 published; v2a C25 threshold x measured ratio; v2a_noaa x 1/0.7; v2b GOES Flare Report both cycles")
    by = []
    for cyc, S in SAMPLES.items():
        for yr, d in S.groupby(S.time.dt.year):
            by.append(dict(cycle=cyc, year=yr, windows=len(d), **{f"pos_{v}": round(d["y_" + v].mean(), 4) for v in VARIANTS if "y_" + v in d}))
    save(pd.DataFrame(by), "audit2_positive_rate_by_year")

run_audit("Audit 2 label variants", audit_labels)


========== Audit 2 label variants ==========


C24 v2b: skipped


C25 v1: FAILED KeyError: 'g16'
C25 v1: skipped
C25 v2a: FAILED KeyError: 'g16'
C25 v2a: skipped
C25 v2a_noaa: FAILED KeyError: 'g16'
C25 v2a_noaa: skipped
C25 v2b: skipped


--> saved audit_v2_out/audit2_label_variants.csv  (v1 published; v2a C25 threshold x measured ratio; v2a_noaa x 1/0.7; v2b GOES Flare Report both cycles)
 cycle  variant  Mplus_flares_with_ar  windows  positive_windows  positive_rate reproduces_v1
    24       v1                   759   215463              7995         0.0371          True
    24      v2a                   759   215463              7995         0.0371              
    24 v2a_noaa                   759   215463              7995         0.0371              
--> saved audit_v2_out/audit2_positive_rate_by_year.csv
 cycle  year  windows  pos_v1  pos_v2a  pos_v2a_noaa
    24  2010    10509  0.0084   0.0084        0.0084
    24  2011    35308  0.0389   0.0389        0.0389
    24  2012    34097  0.0359   0.0359        0.0359
    24  2013    41819  0.0322   0.0322        0.0322
    24  2014    38816  0.0569   0.0569        0.0569
    24  2015    35489  0.0400   0.0400        0.0400
    24  2016     7496  0.0135   0.0135     

## Audits 4–6 — HARP/NOAA overlap, recurrent regions, real date ranges of every split

In [9]:
def splits():
    """The train/test sets used in v1 (same rules, same seeds)."""
    from sklearn.model_selection import GroupKFold
    A, B = SAMPLES[24], SAMPLES[25]
    a_tr = A[A.time < "2014-01-01"]; a_te = A[(A.time >= "2014-01-15") & (~A.ar.isin(set(a_tr.ar)))]
    b_tr = B[B.time < "2023-01-01"]; b_te = B[(B.time >= "2023-01-15") & (~B.ar.isin(set(b_tr.ar)))]
    S = {"C24 forward / A": (24, a_tr, a_te), "C25 in-cycle / C": (25, b_tr, b_te)}
    codes = pd.factorize(A.ar)[0]
    for k, (tr, te) in enumerate(GroupKFold(5).split(A, A.y_v1, codes)):
        S[f"C24 grouped fold {k+1}"] = (24, A.iloc[tr], A.iloc[te])
    return S, (a_tr, a_te, b_tr, b_te)

def audit_harp_noaa():
    rows = []
    for cyc, info in HARPINFO.items():
        n_ar = info.ars.apply(len); ar_h = {}
        for h, s in info.ars.items():
            for a in s: ar_h.setdefault(a, set()).add(h)
        rows.append(dict(cycle=cyc, harps=len(info), harps_with_2plus_noaa=int((n_ar > 1).sum()),
                         noaa_regions=len(ar_h), noaa_in_2plus_harps=int(sum(len(v) > 1 for v in ar_h.values()))))
    save(pd.DataFrame(rows), "audit4_harp_noaa_mapping")
    S, _ = splits(); rows = []
    for name, (cyc, tr, te) in S.items():
        info = HARPINFO[cyc]; tr_h, te_h = set(tr.harp), set(te.harp)
        tr_ars = set().union(*[info.ars[h] for h in tr_h]) if tr_h else set()
        shared = {h for h in te_h if info.ars[h] & tr_ars}
        rows.append(dict(split=name, test_harps=len(te_h), test_harps_sharing_noaa_with_train=len(shared),
                         test_windows=len(te), test_windows_affected=int(te.harp.isin(shared).sum()),
                         test_positives_affected=int(te[te.harp.isin(shared)].y_v1.sum())))
    save(pd.DataFrame(rows), "audit4_split_noaa_overlap", "a test HARP sharing a NOAA number with any training HARP")

def audit_recurrence():
    rows = []
    S, _ = splits()
    for name, (cyc, tr, te) in S.items():
        info = HARPINFO[cyc]; I_tr = info.loc[sorted(set(tr.harp))]; I_te = info.loc[sorted(set(te.harp))]
        n = 0; pairs = []
        for h, r in I_te.iterrows():
            near_lat = np.abs(I_tr.lat - r.lat) <= 5
            # test HARP returns after a training HARP (train -> test) ...
            dt1 = (r.t0 - I_tr.t1).dt.total_seconds() / 86400
            ok1 = (dt1 >= 5) & (dt1 <= 25) & near_lat & (np.abs(((I_tr.carr1 - r.carr0) + 180) % 360 - 180) <= 15)
            # ... or a training HARP returns after this test HARP (test -> train; matters for grouped folds)
            dt2 = (I_tr.t0 - r.t1).dt.total_seconds() / 86400
            ok2 = (dt2 >= 5) & (dt2 <= 25) & near_lat & (np.abs(((I_tr.carr0 - r.carr1) + 180) % 360 - 180) <= 15)
            ok = ok1 | ok2
            if ok.any(): n += 1; pairs.append(f"{h}~{int(ok[ok].index[0])}")
        rows.append(dict(split=name, test_harps=len(I_te), test_harps_likely_recurrent_from_train=n, examples=" ".join(pairs[:5])))
    save(pd.DataFrame(rows), "audit5_recurrent_regions",
         "approximate: one HARP starts 5-25 d after the other ended (either direction), |dlat|<=5 deg, |dCarrington lon|<=15 deg")

def audit_dates():
    _, (a_tr, a_te, b_tr, b_te) = splits()
    rows = []
    for name, d in [("A/D test: C24 forward", a_te), ("A/B train: C24 2010-13", a_tr), ("B/C test: C25 late", b_te), ("C/D train: C25 pool", b_tr)]:
        rows.append(dict(set=name, first=str(d.time.min()), last=str(d.time.max()), windows=len(d), harps=d.harp.nunique(),
                         pos_rate_v1=round(d.y_v1.mean(), 4)))
    save(pd.DataFrame(rows), "audit6_split_dates", "v1 CSV and Figure 2 call the C24 test set '2014-15'")

run_audit("Audit 4 HARP vs NOAA", audit_harp_noaa)
run_audit("Audit 5 recurrent regions", audit_recurrence)
run_audit("Audit 6 split dates", audit_dates)


========== Audit 4 HARP vs NOAA ==========


--> saved audit_v2_out/audit4_harp_noaa_mapping.csv
 cycle  harps  harps_with_2plus_noaa  noaa_regions  noaa_in_2plus_harps
    24   1232                    232          1498                   69
    25    888                    202          1198                   90


[Audit 4 HARP vs NOAA] FAILED — recorded in audit_v2_out/errors.txt
Traceback (most recent call last):
  File "/tmp/ipykernel_2440/2526661432.py", line 26, in run_audit
    fn(); print(f"[{name}] done in {round(time.time()-t0)} s", flush=True)
    ^^^^
  File "/tmp/ipykernel_2440/3028242549.py", line 29, in audit_harp_noaa
    test_positives_affected=int(te[te.harp.isin(shared)].y_v1.sum())))
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/pandas/core/generic.py", line 6434, in __getattr__
    return object.__getattribute__(self, name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'DataFrame' object has no attribute 'y_v1'




========== Audit 5 recurrent regions ==========


--> saved audit_v2_out/audit5_recurrent_regions.csv  (approximate: one HARP starts 5-25 d after the other ended (either direction), |dlat|<=5 deg, |dCarrington lon|<=15 deg)
             split  test_harps  test_harps_likely_recurrent_from_train                                          examples
   C24 forward / A         509                                       4           3586~3483 3601~3497 3635~3513 3688~3560
  C25 in-cycle / C         490                                       7 8950~8873 8953~8868 8961~8868 8990~8903 8995~8927
C24 grouped fold 1         242                                     122           226~187 256~220 316~273 362~392 394~355
C24 grouped fold 2         241                                     111             67~43 190~147 220~185 274~318 279~323
C24 grouped fold 3         241                                     123               43~67 92~67 135~175 146~107 211~175
C24 grouped fold 4         241                                     119           107~146 187~147 273


========== Audit 6 split dates ==========


[Audit 6 split dates] FAILED — recorded in audit_v2_out/errors.txt
Traceback (most recent call last):
  File "/tmp/ipykernel_2440/2526661432.py", line 26, in run_audit
    fn(); print(f"[{name}] done in {round(time.time()-t0)} s", flush=True)
    ^^^^
  File "/tmp/ipykernel_2440/3028242549.py", line 57, in audit_dates
    pos_rate_v1=round(d.y_v1.mean(), 4)))
                      ^^^^^^
  File "/opt/hostedtoolcache/Python/3.12.15/x64/lib/python3.12/site-packages/pandas/core/generic.py", line 6434, in __getattr__
    return object.__getattribute__(self, name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'DataFrame' object has no attribute 'y_v1'



## Provenance, then save everything (zip + Drive + download)

In [10]:
def md5(p, n=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(n), b""): h.update(ch)
    return h.hexdigest()
prov = {"run_finished_utc": pd.Timestamp.utcnow().isoformat(), "versions": vers, "scale_factor": SCALE,
        "M_true": M_TRUE, "M_scaled_equivalent": M_SCALED_EQ, "flux_source": FLUX_SRC, "ratio_used_v2a": RATIO, "errors": len(ERRORS),
        "inputs": {p: {"bytes": os.path.getsize(p), "md5": md5(p)} for p in
                   ["swan/addenda.tar.gz", "pinn_data/sharp_2010_2018_1h.csv.gz", "pinn_data/sharp_2019_2024_1h.csv.gz", "flsum_g16.nc"] + sorted(glob.glob("flrpt/*.nc")) if os.path.exists(p)},
        "hek_cache_files": len(glob.glob("hek_cache/*.csv"))}
json.dump(prov, open(f"{OUT}/provenance.json", "w"), indent=1, default=str)
print(json.dumps(prov, indent=1, default=str))

shutil.make_archive("audit_v2_results", "zip", OUT)
print("\nFiles in", OUT, ":", sorted(os.listdir(OUT)))
print("ERRORS:", len(ERRORS), "— see errors.txt" if ERRORS else "")
# 1) download first (never waits on a popup)
try:
    from google.colab import files; files.download("audit_v2_results.zip"); print("download started")
except Exception as e: print("download skipped:", type(e).__name__)
# 2) copy to Drive only if Drive is already mounted (no new permission popup at the end)
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(f"{BACKUP_V2}/hek_cache", exist_ok=True)
    for p in ["audit_v2_results.zip", "c24_samples_v2.pkl", "c25_samples_v2.pkl"]:
        if os.path.exists(p): shutil.copy(p, f"{BACKUP_V2}/{p}")
    checkpoint(); print("saved to", BACKUP_V2)
else: print("Drive not mounted; results are only in this Colab session and the downloaded zip")

{
 "run_finished_utc": "2026-10-27T12:49:26.641081+00:00",
 "versions": {
  "python": "3.12.15",
  "pandas": "3.1.0",
  "numpy": "2.5.4",
  "scikit-learn": "1.9.1",
  "drms": "0.9.1",
  "xarray": "2026.9.0",
  "netCDF4": "1.7.5.1.post1"
 },
 "scale_factor": 0.7,
 "M_true": 1e-05,
 "M_scaled_equivalent": 1.4285714285714287e-05,
 "flux_source": {},
 "ratio_used_v2a": {
  "g16_over_swan": 1.4285714285714286
 },
 "errors": 6,
 "inputs": {
  "swan/addenda.tar.gz": {
   "bytes": 34042374,
   "md5": "82c4ab7d6593cb34dfb4d9c216767eae"
  },
  "pinn_data/sharp_2010_2018_1h.csv.gz": {
   "bytes": 53760263,
   "md5": "d939b79cc42453d26dcb82d16fc20669"
  },
  "pinn_data/sharp_2019_2024_1h.csv.gz": {
   "bytes": 36783515,
   "md5": "29190875f4ac190721b373b15115c3c4"
  },
  "flrpt/sci_xrsf-l2-flrpt_geo_y2010_v1-0-1.nc": {
   "bytes": 2298466,
   "md5": "352e921b203c9dfe69743de571caf88b"
  },
  "flrpt/sci_xrsf-l2-flrpt_geo_y2011_v1-0-1.nc": {
   "bytes": 3468586,
   "md5": "329591fb3745702541df72782c5